In [ ]:
import sys
from pathlib import Path

taichi_root = Path('/home/fs72041/olgaridzel/dev/taichi_gpu_173')
if not (taichi_root / 'taichi' / '__init__.py').is_file():
    raise FileNotFoundError(f'Check the working Taichi installation: {taichi_root}')
sys.path.insert(0, str(taichi_root))

# Optional: use a source checkout if seemc-taichi is not installed in this kernel.
# Change this path if your project is elsewhere.
seemc_project = Path('/home/fs72041/olgaridzel/dev/seemc-taichi-v0.9.3')
if (seemc_project / 'src' / 'seemc_taichi').is_dir():
    sys.path.insert(1, str(seemc_project / 'src'))

import taichi as ti
print('Python:', sys.executable)
print('Taichi:', ti.__version__)
print('Loaded from:', ti.__file__)
if not Path(ti.__file__).resolve().is_relative_to(taichi_root.resolve()):
    raise RuntimeError('A different Taichi was already imported. Restart the kernel and run this cell first.')


In [ ]:
import sys
import taichi as ti

print("Python:", sys.executable)
print("Taichi:", ti.__version__)

ti.init(
    arch=ti.cuda,
    default_fp=ti.f64,
    fast_math=False,
)

print("Taichi arch:", ti.cfg.arch)

x = ti.field(dtype=ti.f64, shape=10_000_000)

@ti.kernel
def fill():
    for i in x:
        x[i] = i * 0.123456789

fill()
ti.sync()

print("CUDA test complete:", x[123])

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas
import matplotlib.pyplot as plt

import rfa_model as rfa

In [ ]:
# Apply CUDA dispatch before calling rfa.build_rfa_field.
# Put rfa_cuda_sor_patch.py (v3) alongside this notebook.
from rfa_cuda_sor_patch import enable_cuda_sor, smoke_test_cuda
import rfa_model.taichi_sor as _rfa_ts

enable_cuda_sor()
assert getattr(_rfa_ts.solve_red_black_sor_taichi, "_rfa_cuda_patch", False)
smoke_test_cuda()  # 17^3 analytic Laplace test; do not proceed if it fails.


In [ ]:
%matplotlib widget

In [ ]:
from rfa_model.io import save_field_npz


def voltage_tag(value, unit="V"):
    """
    Convert voltage value to filename-safe tag.

    Examples:
        50.0  -> p50V
        -50.0 -> m50V
        0.0   -> p0V
        2.5   -> p2p5V
    """
    value = float(value)

    sign = "m" if value < 0 else "p"
    mag = abs(value)

    # Use integer form when possible.
    if mag.is_integer():
        s = str(int(mag))
    else:
        s = f"{mag:g}".replace(".", "p")

    return f"{sign}{s}{unit}"


def field_filename_from_voltages(
    voltages: dict,
    h: float,
    prefix: str = "rfa_field",
    out_dir: str | Path = "data",
    extra_tag: str | None = None,
):
    """
    Build a descriptive filename from h and voltage dictionary.
    """
    out_dir = Path(out_dir)

    h_mm = 1e3 * float(h)

    if h_mm.is_integer():
        h_tag = f"h{int(h_mm)}mm"
    else:
        h_tag = f"h{h_mm:g}mm".replace(".", "p")

    parts = [
        prefix,
        h_tag,
        f"Vs{voltage_tag(voltages.get('Vs', 0.0))}",
        f"Vg1{voltage_tag(voltages.get('Vg1', 0.0))}",
        f"Vg2{voltage_tag(voltages.get('Vg2', 0.0))}",
        f"Vg3{voltage_tag(voltages.get('Vg3', 0.0))}",
        f"Vc{voltage_tag(voltages.get('Vc', 0.0))}",
        f"Vdt{voltage_tag(voltages.get('Vdt', 0.0))}",
    ]

    if "Vr" in voltages:
        parts.append(f"Vr{voltage_tag(voltages['Vr'])}")

    if extra_tag is not None:
        parts.append(str(extra_tag))

    filename = "_".join(parts) + ".npz"

    return out_dir / filename

In [ ]:
stl_dir = Path("rfa stl")

meshes, sample_parts = rfa.load_and_align_sample_assembly(
    stl_dir,
    alpha_deg=0.0,
)

frame_meshes, frame_parts, frame_info = rfa.load_and_align_grid_frames(
    stl_dir,
    verbose=True,
)

sample_y_bounds, sample_z_bounds = rfa.sample_bounds(meshes)

In [ ]:
voltages = {
    "Vs":  50.0,
    "Vg1": 0.0,
    "Vg2": 0.0,
    "Vg3": 0.0,
    "Vc": 50.0,
    "Vdt": 0.0,
}


field = rfa.build_rfa_field(
    xyz_min=(-0.083, -0.083, -0.083),
    xyz_max=(0.083, 0.083, 0.083),
    h=0.2e-3,
    voltages=voltages,
    meshes=meshes,
    frame_meshes=frame_meshes,
    R_g1=frame_info["R_g1"],
    R_g2=frame_info["R_g2"],
    R_g3=frame_info["R_g3"],
    R_col=0.08255,
    mesh_method="voxelized",
    solver="taichi_sor",
    taichi_arch="cuda",
    taichi_precision="f64",
    taichi_check_every=25,
    taichi_restore_best_on_max_iter=True,
    max_iter=10_000,
    tol=7.3e-5,
    omega=1.85,
    verbose=True,
)

Ex_interp, Ey_interp, Ez_interp = rfa.build_field_interpolators(field)
Phi_interp = rfa.build_potential_interpolator(field)

In [ ]:
from rfa_model.io import save_field_npz

save_field_npz(
    field,
    "fields/field_h02mm_Vs50_Vg20_Vg30_Vc50_alpha0.npz",
)

In [ ]:
x = np.asarray(field["x"])
y = np.asarray(field["y"])
z = np.asarray(field["z"])
iz = np.argmin(np.abs(z))

fig, ax = plt.subplots(figsize=(9, 4.5))

for y_value_mm in [0, 10, 20]:
    iy = np.argmin(np.abs(y - y_value_mm * 1e-3))
    ax.plot(
        x * 1e3,
        field["V"][:, iy, iz],
        label=f"y = {y_value_mm} mm",
    )

ax.set_xlabel("x (mm)")
ax.set_ylabel("Potential (V)")
ax.grid(True)
ax.legend()
plt.show()

In [ ]:
from field_validation import (
    validate_field_numerics,
    print_validation_report,
    plot_field_overview,
)

report = validate_field_numerics(field)
print_validation_report(report)

if "fig" in globals():
    plt.close(fig)

fig, axes = plot_field_overview(
    field,
    plane="xy",
    coordinate=0.0,
)

fig.canvas.draw()
display(fig)
plt.savefig("field_xy_slice_h02mm_BSEY_RG_alpha_30.png")

In [ ]:
for name, mesh in frame_meshes.items():
    print(
        f"{name:18s}",
        f"watertight={mesh.is_watertight}",
        f"winding={mesh.is_winding_consistent}",
        f"bodies={mesh.body_count}",
    )

In [ ]:
field = rfa.load_field_npz("data/field_h02mm_Vs0_Vg2m50_Vg3m50_Vc50_alpha85.npz")

print(field.keys())
print("V shape:", field["V"].shape)
print("h:", field["h"])

In [ ]:
import importlib
import field_validation

importlib.reload(field_validation)

In [ ]:
from field_validation import (
    plot_presentation_cutaway,
    plot_presentation_potential_and_field,
)

if "fig1" in globals():
    plt.close(fig1)

# 1. Electrostatic cutaway with field lines and active-region inset
fig1, ax1 = plot_presentation_cutaway(
    field,
    plane="xy",
    coordinate=0.0,
)

# 2. Potential + log10 field-strength comparison
# fig2, axes2 = plot_presentation_potential_and_field(
#     field,
#     plane="xy",
#     coordinate=0.0,
# )

fig1.savefig("rfa_field_bsey_rg_cutaway_alpha85_xy.png", dpi=600, bbox_inches="tight")
# fig2.savefig("rfa_potential_and_field_bsey_rg_alpha30.png", dpi=600, bbox_inches="tight")

In [ ]:
print("Method:", field["mesh_voxel_methods"].get("rod"))
print("Watertight:", field["mesh_watertight"].get("rod"))
print("Connectivity:", field["mesh_connectivity"].get("rod"))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

owner = np.asarray(field["owner"])
x = np.asarray(field["x"])
y = np.asarray(field["y"])
z = np.asarray(field["z"])

rod_id = field["owner_id_map"]["rod"]
rod = owner == rod_id

iy0 = np.argmin(np.abs(y))
half_width = 3

lo = max(0, iy0 - half_width)
hi = min(len(y), iy0 + half_width + 1)

rod_exact = rod[:, iy0, :].T
rod_slab = np.any(rod[:, lo:hi, :], axis=1).T

extent = [
    x[0] * 1e3, x[-1] * 1e3,
    z[0] * 1e3, z[-1] * 1e3,
]

fig, axes = plt.subplots(1, 2, figsize=(10, 8), sharex=True, sharey=True)

axes[0].imshow(
    rod_exact,
    origin="lower",
    extent=extent,
    interpolation="nearest",
    aspect="equal",
)
axes[0].set_title(f"Exact rod slice: y={y[iy0]*1e3:.2f} mm")

axes[1].imshow(
    rod_slab,
    origin="lower",
    extent=extent,
    interpolation="nearest",
    aspect="equal",
)
axes[1].set_title(
    f"Rod projection: y={y[lo]*1e3:.2f} to {y[hi-1]*1e3:.2f} mm"
)

for ax in axes:
    ax.set_xlim(-20, 20)
    ax.set_ylim(-85, 15)
    ax.set_xlabel("x (mm)")
axes[0].set_ylabel("z (mm)")

plt.show()

In [ ]:
rod_full_projection = np.any(rod, axis=1).T

fig, ax = plt.subplots(figsize=(6, 9))
ax.imshow(
    rod_full_projection,
    origin="lower",
    extent=[
        x[0] * 1e3, x[-1] * 1e3,
        z[0] * 1e3, z[-1] * 1e3,
    ],
    interpolation="nearest",
    aspect="equal",
)

ax.set_xlim(-20, 20)
ax.set_ylim(-85, 15)
ax.set_xlabel("x (mm)")
ax.set_ylabel("z (mm)")
ax.set_title("Rod projection through full y extent")
plt.show()

In [ ]:
from rfa_model.omega_sweep import (
    run_omega_sweep,
    recommended_omega,
    save_omega_sweep_csv,
)

results = run_omega_sweep(
    field,
    omegas=[1.85, 1.90, 1.94, 1.97, 1.98, 1.985],
    iterations=750,
    check_every=25,
    arch="metal",
    precision="f32",
)

best_omega = recommended_omega(results)
print("Recommended omega:", best_omega)

save_omega_sweep_csv(results, "omega_sweep.csv")